<a href="https://colab.research.google.com/github/karenl7/engr550-internal/blob/main/03_rocket_landing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Rocket Landing

In this exercise, you will use **sequential quadratic programming (SQP)** to plan a landing trajectory for a nonlinear rocket model. You will implement the dynamics, explore a PD controller, and then solve a sequence of quadratic programs to improve the landing trajectory. Finally, you will incorporate obstacle avoidance and explore the use of slack variables to soften state constraints.

The rocket moves in a vertical plane. Its model consists of a thruster base with mass $m_t$ and a rigid rocket body with mass $m_b$ with length $\ell$, which can rotate about the thruster base. This model resembles an inverted cart-pole, with additional thrust inputs that allow vertical motion and attitude control.

The **state vector** is
$$
x = [p_x,\ p_z,\ \theta,\ \dot p_x,\ \dot p_z,\ \dot\theta]^\top,
$$
where $(p_x,p_z)$ is the position of the thruster base, $\theta$ is the clockwise angle from vertical of the rocket body, and $(\dot p_x,\dot p_z,\dot\theta)$ are the corresponding linear and angular velocities. The upright configuration is $\theta=0$.

The rocket as three thrusters. There are two thrusters at the thruster base which apply horizontal and vertical forces, and there is one thruster at the body body nose tip which applies a horizontal force, producing a moment about the base.
The resulting **control vector** is
$$
u = [T_{\mathrm{base},x},\ T_{\mathrm{base},z},\ T_{\mathrm{nose},x}]^\top.
$$


Your goal is to bring the rocket from its initial position and velocity to the landing pad, upright and at rest. The trajectory optimization balances state and control costs with a terminal cost that penalizes error relative to this target. You will enforce actuator limits and altitude constraints, then add a circular no-fly zone and examine how constraint softening affects the solution.



In [ ]:
!pip install equinox

In [ ]:
import functools
import os
import warnings
from typing import Callable, Dict, List

import cvxpy as cp
import cvxpy_debug as cp_debug
import dynamaxsys
import equinox as eqx

# import jax and jax.numpy. No regular numpy needed anymore!
import jax
import jax.numpy as jnp
import matplotlib.animation as animation
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon

import numpy as np
from IPython.display import HTML, display
from functools import wraps


Helper functions for plotting and visualization. Just run this cell. You can also minimize the cell so it doesn't add clutter to your notebook.


In [ ]:
def plot_linearization(
    diagnostic_A_sequence,
    diagnostic_B_sequence,
    diagnostic_C_sequence,
    diagnostic_state_history,
    diagnostic_control_history,
):
    """Overlay recorded and predicted one-step motion on the rocket trajectory."""
    x = np.asarray(diagnostic_state_history)
    u = np.asarray(diagnostic_control_history)
    A = np.asarray(diagnostic_A_sequence)
    B = np.asarray(diagnostic_B_sequence)
    C = np.asarray(diagnostic_C_sequence)

    n_steps = len(u)
    if x.shape != (n_steps + 1, 6) or u.shape != (n_steps, 3):
        raise ValueError("Expected states (N+1, 6) and controls (N, 3).")
    if A.shape != (n_steps, 6, 6) or B.shape != (n_steps, 6, 3) or C.shape != (n_steps, 6):
        raise ValueError("Expected A (N, 6, 6), B (N, 6, 3), and C (N, 6).")

    predicted_next = np.array([
        A[k] @ x[k] + B[k] @ u[k] + C[k]
        for k in range(n_steps)
    ])

    # Space out the rocket drawings while keeping arrows at every step.
    ax = plot_rocket_trajectory(x, stride=max(1, n_steps // 15), alpha=0.15)

    origin = x[:-1, :2]
    for endpoint, color, width, label in (
        (x[1:, :2],          "tab:blue",   0.006, "Recorded next position"),
        (predicted_next[:, :2], "tab:orange", 0.003, "Linearized next position"),
    ):
        delta = endpoint - origin
        ax.quiver(
            origin[:, 0], origin[:, 1], delta[:, 0], delta[:, 1],
            angles="xy", scale_units="xy", scale=1, pivot="tail",
            color=color, width=width, minlength=0, zorder=5, label=label,
        )
        ax.update_datalim(endpoint)

    ax.autoscale_view()
    ax.legend()
    return ax

# run cell as is.
# plotting and animation code
def _with_transparency_arg(fn):
    @wraps(fn)
    def wrapper(
        ax: plt.Axes,
        state: jnp.ndarray,
        control: jnp.ndarray | None = None,
        color: str = "C0",
        label: str | None = None,
        rocket_length: float = 2.0,
        thrust_limit: List[float] = [100.0, 300.0, 100.0],
        transparency: float = 0.6,
    ):
        # Capture current artists so only newly drawn rocket elements are updated.
        p0, l0, c0 = len(ax.patches), len(ax.lines), len(ax.collections)

        out = fn(
            ax=ax,
            state=state,
            control=control,
            color=color,
            label=label,
            rocket_length=rocket_length,
            thrust_limit=thrust_limit,
        )

        t = float(np.clip(transparency, 0.0, 1.0))
        factor = t / 0.6  # 0.6 is the previous baseline alpha in the drawing.

        def _scale_alpha(artist):
            if artist.get_zorder() < 3:  # keep background/ground unchanged
                return
            a = artist.get_alpha()
            a = 1.0 if a is None else float(a)
            artist.set_alpha(float(np.clip(a * factor, 0.0, 1.0)))

        for art in ax.patches[p0:]:
            _scale_alpha(art)
        for art in ax.lines[l0:]:
            _scale_alpha(art)
        for art in ax.collections[c0:]:
            _scale_alpha(art)

        return out

    return wrapper

@_with_transparency_arg
def plot_rocket(
    ax: plt.Axes,
    state: jnp.ndarray,
    control: jnp.ndarray | None = None,
    color: str = "C0",
    label: str | None = None,
    rocket_length: float = 2.0,
    thrust_limit: List[float] = [100.0, 300.0, 100.0],
):
    """Visualize rocket body (base->nose) and optional control vectors."""
    x, z, theta = float(state[0]), float(state[1]), float(state[2])

    # Rocket body direction (theta measured clockwise from vertical)
    dx = rocket_length * np.sin(theta)
    dz = rocket_length * np.cos(theta)

    x_base, z_base = x, z
    x_nose, z_nose = x + dx, z + dz

    # Draw rocket body
    # Rocket body as a polygon: rectangular shaft + triangular nose tip
    ux, uz = dx / rocket_length, dz / rocket_length              # unit axis (base -> nose)
    px, pz = uz, -ux                                             # unit perpendicular

    body_half_width = 0.08 * rocket_length
    nose_len = 0.22 * rocket_length

    # End of rectangular shaft (before nose tip)
    x_shaft = x_nose - nose_len * ux
    z_shaft = z_nose - nose_len * uz

    # Shaft corners
    left_base = [x_base + body_half_width * px, z_base + body_half_width * pz]
    right_base = [x_base - body_half_width * px, z_base - body_half_width * pz]
    right_shaft = [x_shaft - body_half_width * px, z_shaft - body_half_width * pz]
    left_shaft = [x_shaft + body_half_width * px, z_shaft + body_half_width * pz]

    # Draw shaft
    ax.add_patch(
        Polygon(
            [left_base, right_base, right_shaft, left_shaft],
            closed=True,
            facecolor=color,
            edgecolor=color,
            linewidth=1.5,
            alpha=0.6,
            label=label,
            zorder=3,
        )
    )

    # Draw triangular nose
    ax.add_patch(
        Polygon(
            [left_shaft, [x_nose, z_nose], right_shaft],
            closed=True,
            facecolor=color,
            edgecolor=color,
            linewidth=1.5,
            alpha=0.6,
            zorder=3.1,
        )
    )

    thrust_limit_arr = np.asarray(thrust_limit, dtype=float)
    base_ref_len = 0.45 * rocket_length
    len_scale = thrust_limit_arr / thrust_limit_arr.max()

    max_bar_len_base_x, max_bar_len_base_z, max_bar_len_nose_x = base_ref_len * len_scale
    # Draw thrust "bars" (length proportional to actuator usage)

    if control is not None:
        T_base_x, T_base_z, T_nose_x = map(float, control)

        # Fixed maximum bar length, scaled to rocket size
        max_bar_len = 0.45 * rocket_length

        # Normalize by actuator limits and clip to [-1, 1]
        base_x_ratio = np.clip(T_base_x / thrust_limit[0], -1.0, 1.0)
        base_z_ratio = np.clip(T_base_z / thrust_limit[1], -1.0, 1.0)
        nose_x_ratio = np.clip(T_nose_x / thrust_limit[2], -1.0, 1.0)

        # Convert normalized values to bar lengths
        base_dx = max_bar_len_base_x * base_x_ratio
        base_dz = max_bar_len_base_z * base_z_ratio
        nose_dx = max_bar_len_nose_x * nose_x_ratio

        # Base thrust bars separated into x and z components
        ax.plot(
            [x_base, x_base + base_dx],
            [z_base, z_base],
            color="crimson",
            linewidth=8,
            solid_capstyle="round",
            zorder=5,
        )
        ax.plot(
            [x_base, x_base + jnp.sign(base_dx) * max_bar_len_base_x],
            [z_base, z_base],
            color="white",
            linewidth=8,
            solid_capstyle="round",
            zorder=4,
        )
        ax.plot(
            [x_base, x_base + jnp.sign(base_dx) * max_bar_len_base_x],
            [z_base, z_base],
            color="crimson",
            linewidth=8,
            alpha=0.4,
            solid_capstyle="round",
            zorder=4,
        )
        # Base thrust (z component): vector + full-scale reference bar
        ax.plot(
            [x_base, x_base],
            [z_base, z_base + base_dz],
            color="crimson",
            linewidth=8,
            solid_capstyle="round",
            zorder=5,
        )
        ax.plot(
            [x_base, x_base],
            [z_base, z_base + jnp.sign(base_dz) * max_bar_len_base_z],
            color="white",
            linewidth=8,
            solid_capstyle="round",
            zorder=4,
        )
        ax.plot(
            [x_base, x_base],
            [z_base, z_base + jnp.sign(base_dz) * max_bar_len_base_z],
            color="crimson",
            linewidth=8,
            alpha=0.4,
            solid_capstyle="round",
            zorder=4,
        )

        # Nose thrust (x component): vector + full-scale reference bar
        ax.plot(
            [x_nose, x_nose + nose_dx],
            [z_nose, z_nose],
            color="crimson",
            linewidth=8,
            solid_capstyle="round",
            zorder=5,
        )
        ax.plot(
            [x_nose, x_nose + jnp.sign(nose_dx) * max_bar_len_nose_x],
            [z_nose, z_nose],
            color="white",
            linewidth=8,
            solid_capstyle="round",
            zorder=4,
        )
        ax.plot(
            [x_nose, x_nose + jnp.sign(nose_dx) * max_bar_len_nose_x],
            [z_nose, z_nose],
            color="crimson",
            linewidth=8,
            alpha=0.4,
            solid_capstyle="round",
            zorder=4,
        )

    # Ground line for context
    ax.hlines(0.0, -3.0, 3.0, color="k", linewidth=2, zorder=1)
    ax.set_aspect("equal")



def plot_rocket_trajectory(
    states,
    controls=None,
    ax=None,
    stride=1,
    alpha=0.15,
    color="C0",
    rocket_length=2.0,
    thrust_limit=(200.0, 300.0, 100.0),
):
    """Overlay translucent rocket poses along a state sequence.

    Args:
        states: Array of shape (N, 6), in the usual rocket state order.
        controls: Optional array of shape (N, 3) or (N-1, 3). For N-1
            controls, the final pose is drawn without thrust vectors.
        ax: Optional existing Matplotlib axis.
        stride: Draw every `stride` states, always including the final state.
        alpha: Opacity of each pose and its controls, from 0 to 1.
        color: Rocket body color.
        rocket_length: Rocket length in meters.
        thrust_limit: Three actuator limits used by `plot_rocket`.

    Returns:
        The Matplotlib axis, for further annotations or axis adjustments.
    """
    states = np.asarray(states)
    if states.ndim != 2 or states.shape[1] != 6 or len(states) == 0:
        raise ValueError("states must have shape (N, 6), with N >= 1")
    if not isinstance(stride, (int, np.integer)) or stride < 1:
        raise ValueError("stride must be a positive integer")
    if not 0 <= alpha <= 1:
        raise ValueError("alpha must be between 0 and 1")
    if controls is not None:
        controls = np.asarray(controls)
        if controls.shape not in ((len(states), 3), (len(states) - 1, 3)):
            raise ValueError("controls must have shape (N, 3) or (N-1, 3)")

    if ax is None:
        _, ax = plt.subplots(figsize=(7, 7))

    indices = np.unique(np.r_[np.arange(0, len(states), stride), len(states) - 1])
    for i in indices:
        control = controls[i] if controls is not None and i < len(controls) else None
        previous_artists = set(ax.get_children())
        # Some versions of plot_rocket wrap ax.quiver; keep wrappers from accumulating.
        original_quiver = ax.quiver
        try:
            plot_rocket(ax, states[i], control=control, color=color,
                        rocket_length=rocket_length, thrust_limit=thrust_limit)
        finally:
            ax.quiver = original_quiver
        for artist in set(ax.get_children()) - previous_artists:
            if artist.get_zorder() < 3 and artist in ax.collections:
                # Remove the repeated ground line drawn by plot_rocket.
                artist.remove()
            else:
                artist.set_alpha(alpha)

    ax.axhline(0, color="k", linewidth=1.5, zorder=0)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("Horizontal position [m]")
    ax.set_ylabel("Altitude [m]")
    ax.set_title("Rocket poses along the trajectory")
    ax.grid(alpha=0.2)
    ax.autoscale_view()
    return ax

# def plot_abc(
#     as_input,
#     bs_input,
# ):
#     pass

def animate_rocket_trajectory(
    states,
    controls=None,
    dt=0.01,
    rocket_length=2.0,
    thrust_limit=(200.0, 300.0, 100.0),
    obstacle_center=None,
    obstacle_radius=None,
    stride=1,
    speed=1.0,
    repeat=False,
    show=True,
):
    """Animate a rocket trajectory using the same geometry as ``plot_rocket``.

    Args:
        states: NumPy/JAX array of shape (N, 6), ordered as
            [p_x, p_z, theta, v_x, v_z, omega]. Angles are in radians.
        controls: Optional (N, 3) or (N-1, 3) array, ordered as
            [T_base_x, T_base_z, T_nose_x]. With N-1 controls, the
            terminal pose is shown without thrust indicators.
        dt: Simulation time between consecutive states, in seconds.
        rocket_length: Rocket length in meters; use the dynamics model's length.
        thrust_limit: Three positive actuator magnitudes for thrust indicators.
        obstacle_center: Optional (x, z) center of a circular no-fly zone.
        obstacle_radius: Radius in meters; supply together with obstacle_center.
        stride: Show every stride-th state, always including the terminal state.
        speed: Playback speed multiplier (1.0 plays at simulation speed).
        repeat: Whether playback loops.
        show: Return inline HTML if True, or a Matplotlib FuncAnimation if False.

    Examples:
        animate_rocket_trajectory(states, controls, dt=dt,
                                 rocket_length=rocket_length, thrust_limit=upper)
        # To save a GIF (requires Pillow):
        anim = animate_rocket_trajectory(states, controls, show=False)
        anim.save("landing.gif", writer="pillow", fps=100)
    """
    states = np.asarray(states, dtype=float)
    if states.ndim != 2 or states.shape[1] != 6 or len(states) == 0:
        raise ValueError("states must have shape (N, 6), with N >= 1")
    if not np.all(np.isfinite(states)):
        raise ValueError("states must contain only finite values")
    if controls is not None:
        controls = np.asarray(controls, dtype=float)
        if controls.shape not in ((len(states), 3), (len(states) - 1, 3)):
            raise ValueError("controls must have shape (N, 3) or (N-1, 3)")
        if not np.all(np.isfinite(controls)):
            raise ValueError("controls must contain only finite values")
    if not isinstance(stride, (int, np.integer)) or stride < 1:
        raise ValueError("stride must be a positive integer")
    if any(not np.isfinite(v) or v <= 0 for v in (dt, speed, rocket_length)):
        raise ValueError("dt, speed, and rocket_length must be positive and finite")
    thrust_limit = np.asarray(thrust_limit, dtype=float)
    if (thrust_limit.shape != (3,) or not np.all(np.isfinite(thrust_limit))
            or np.any(thrust_limit <= 0)):
        raise ValueError("thrust_limit must contain three positive, finite values")
    if (obstacle_center is None) != (obstacle_radius is None):
        raise ValueError("supply both obstacle_center and obstacle_radius")
    if obstacle_center is not None:
        obstacle_center = np.asarray(obstacle_center, dtype=float)
        if obstacle_center.shape != (2,) or not np.all(np.isfinite(obstacle_center)):
            raise ValueError("obstacle_center must contain two finite coordinates")
        if not np.isfinite(obstacle_radius) or obstacle_radius <= 0:
            raise ValueError("obstacle_radius must be positive and finite")

    fig, ax = plt.subplots(figsize=(15, 15))
    ax.axhline(0.0, color="k", linewidth=1.5, zorder=0) # ground line
    ax.plot(0.0, 0.0, "s", color="tab:green", markersize=5, label="Landing pad") # landing pad
    ax.plot(states[:, 0], states[:, 1], "--", color="C0", alpha=0.2) # planned trajectory
    trail, = ax.plot([], [], color="C0", alpha=0.7, linewidth=1.5)
    clock = ax.text(0.02, 0.97, "", transform=ax.transAxes, va="top")

    # Fixed limits include every base/nose pose, the pad, and any obstacle.
    noses = states[:, :2] + rocket_length * np.column_stack(
        (np.sin(states[:, 2]), np.cos(states[:, 2]))
    )
    bounds = np.vstack((states[:, :2], noses, [[0.0, 0.0]]))
    if obstacle_center is not None:
        ax.add_patch(plt.Circle(obstacle_center, obstacle_radius,
                                color="tab:red", alpha=0.2, label="No-fly zone"))
        bounds = np.vstack((bounds, obstacle_center - obstacle_radius,
                            obstacle_center + obstacle_radius))
    margin = max(0.6 * rocket_length, 0.05 * np.ptp(bounds, axis=0).max())
    ax.set_xlim(bounds[:, 0].min() - margin, bounds[:, 0].max() + margin)
    ax.set_ylim(bounds[:, 1].min() - margin, bounds[:, 1].max() + margin)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("Horizontal position [m]")
    ax.set_ylabel("Altitude [m]")
    ax.set_title("Rocket landing")
    ax.grid(alpha=0.2)
    ax.legend(loc="upper right")
    fig.tight_layout()

    rocket_artists = []
    frame_indices = np.unique(np.r_[np.arange(0, len(states), stride), len(states) - 1])

    def update(frame):
        i = int(frame_indices[frame])
        for artist in rocket_artists:
            artist.remove()
        rocket_artists.clear()
        previous_artists = set(ax.get_children())
        control = controls[i] if controls is not None and i < len(controls) else None
        plot_rocket(ax, states[i], control=control, rocket_length=rocket_length,
                    thrust_limit=thrust_limit)
        for artist in set(ax.get_children()) - previous_artists:
            if artist in ax.collections and artist.get_zorder() < 3:
                artist.remove()  # plot_rocket also draws a ground line.
            else:
                rocket_artists.append(artist)
        trail.set_data(states[:i + 1, 0], states[:i + 1, 1])
        clock.set_text(f"t = {i * dt:.2f} s")
        return (*rocket_artists, trail, clock)

    anim = animation.FuncAnimation(
        fig, update, frames=len(frame_indices), init_func=lambda: update(0),
        interval=1000.0 * dt * stride / speed, blit=False, repeat=repeat,
    )
    if show:
        try:
            return HTML(anim.to_jshtml())
        finally:
            plt.close(fig)
    plt.close(fig)  # Avoid a duplicate static figure in a notebook.
    return anim


def animate_sqp_iterations(
    states_list,
    controls_list=None,
    costs=None,
    rocket_length=2.0,
    thrust_limit=(200.0, 300.0, 100.0),
    target_state=None,
    obstacle_center=None,
    obstacle_radius=None,
    interval=400,
    iteration_stride=1,
    num_poses=8,
    repeat=False,
    show=True,
    rocket_alpha=0.25,
):
    """Show how the complete landing trajectory changes across SQP iterations.

    Each frame is one SQP iterate, rather than a time step of the flight.
    Pass NumPy/JAX arrays ``states_list`` of shape (K, N, 6) and optionally
    ``controls_list`` of shape (K, N-1, 3) or (K, N, 3). Index 0 is the
    initial guess, matching the SQP histories collected in this notebook.

    ``costs`` may have K entries, or K-1 entries if no objective was recorded
    for the initial guess. The plotted QP objective includes any trust-region
    and slack penalties; it need not decrease monotonically and is not the
    cost of the nonlinear rollout. Landing errors use the supplied rollouts.

    ``target_state`` defaults to six zeros. ``interval`` is milliseconds per
    iteration. ``iteration_stride`` skips iterates but includes the last one;
    ``num_poses`` controls the number of rocket poses on each trajectory
    (0 hides them). Limits stay fixed across all iterates for comparison.
    Supply both obstacle parameters to draw a circular no-fly zone.
    ``rocket_alpha`` sets rocket opacity (0 = invisible, 1 = opaque).

    Returns inline HTML with playback controls, or a FuncAnimation when
    ``show=False`` (for example, to save with the Pillow GIF writer).

    Example:
        animate_sqp_iterations(states_list, controls_list, costs=costs,
                               rocket_length=rocket_length, thrust_limit=upper)
    """
    states = np.asarray(states_list, dtype=float)
    if (states.ndim != 3 or states.shape[2] != 6
            or states.shape[0] == 0 or states.shape[1] == 0):
        raise ValueError("states_list must have shape (K, N, 6), with K, N >= 1")
    if not np.all(np.isfinite(states)):
        raise ValueError("states_list must contain only finite rollout states")
    K, N, _ = states.shape
    controls = None if controls_list is None else np.asarray(controls_list, dtype=float)
    if controls is not None:
        if controls.shape not in ((K, N - 1, 3), (K, N, 3)):
            raise ValueError("controls_list must have shape (K, N-1, 3) or (K, N, 3)")
        if not np.all(np.isfinite(controls)):
            raise ValueError("controls_list must contain only finite values")
    for name, value, minimum in (("iteration_stride", iteration_stride, 1),
                                  ("num_poses", num_poses, 0)):
        if not isinstance(value, (int, np.integer)) or value < minimum:
            raise ValueError(f"{name} must be an integer >= {minimum}")
    if not np.isfinite(rocket_alpha) or not 0 <= rocket_alpha <= 1:
        raise ValueError("rocket_alpha must be between 0 and 1")
    if not np.isfinite(interval) or interval <= 0:
        raise ValueError("interval must be positive and finite")
    if not np.isfinite(rocket_length) or rocket_length <= 0:
        raise ValueError("rocket_length must be positive and finite")
    thrust_limit = np.asarray(thrust_limit, dtype=float)
    if (thrust_limit.shape != (3,) or not np.all(np.isfinite(thrust_limit))
            or np.any(thrust_limit <= 0)):
        raise ValueError("thrust_limit must contain three positive, finite values")
    target = np.zeros(6) if target_state is None else np.asarray(target_state, dtype=float)
    if target.shape != (6,) or not np.all(np.isfinite(target)):
        raise ValueError("target_state must contain six finite values")
    if (obstacle_center is None) != (obstacle_radius is None):
        raise ValueError("supply both obstacle_center and obstacle_radius")
    if obstacle_center is not None:
        obstacle_center = np.asarray(obstacle_center, dtype=float)
        if obstacle_center.shape != (2,) or not np.all(np.isfinite(obstacle_center)):
            raise ValueError("obstacle_center must contain two finite coordinates")
        if not np.isfinite(obstacle_radius) or obstacle_radius <= 0:
            raise ValueError("obstacle_radius must be positive and finite")
    objective = None
    if costs is not None:
        objective = np.asarray(costs, dtype=float)
        if objective.shape not in ((K,), (K - 1,)):
            raise ValueError("costs must have K or K-1 entries")
        if not np.all(np.isfinite(objective)):
            raise ValueError("costs must contain only finite values")
        if len(objective) == K - 1:
            objective = np.r_[np.nan, objective]

    if objective is None:
        fig, ax = plt.subplots(figsize=(8, 10))
        cost_ax = None
    else:
        fig, (ax, cost_ax) = plt.subplots(1, 2, figsize=(12, 10),
                                          gridspec_kw={"width_ratios": [1.5, 1]})
    ax.axhline(0.0, color="k", linewidth=1.5, zorder=0)
    ax.plot(target[0], target[1], "s", color="tab:green", markersize=8,
            label="Landing target")
    ax.plot(states[0, :, 0], states[0, :, 1], ":", color="0.5",
            alpha=0.6, label="Initial guess")
    previous_line, = ax.plot([], [], "--", color="0.6", label="Previous iterate")
    current_line, = ax.plot([], [], color="C0", linewidth=2, label="Current iterate")
    endpoint, = ax.plot([], [], "o", color="C0", markersize=6)
    status = ax.text(0.02, 0.98, "", transform=ax.transAxes, va="top",
                     fontsize=9, zorder=10,
                     bbox=dict(facecolor="white", alpha=0.85, edgecolor="none"))

    noses = states[:, :, :2] + rocket_length * np.stack(
        (np.sin(states[:, :, 2]), np.cos(states[:, :, 2])), axis=-1
    )
    bounds = np.vstack((states[:, :, :2].reshape(-1, 2), noses.reshape(-1, 2),
                        target[None, :2], [[target[0], 0.0]]))
    if obstacle_center is not None:
        ax.add_patch(plt.Circle(obstacle_center, obstacle_radius, color="tab:red",
                                alpha=0.2, label="No-fly zone"))
        bounds = np.vstack((bounds, obstacle_center - obstacle_radius,
                            obstacle_center + obstacle_radius))
    margin = max(0.6 * rocket_length, 0.05 * np.ptp(bounds, axis=0).max())
    ax.set_xlim(bounds[:, 0].min() - margin, bounds[:, 0].max() + margin)
    ax.set_ylim(bounds[:, 1].min() - margin, bounds[:, 1].max() + margin)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("Horizontal position [m]")
    ax.set_ylabel("Altitude [m]")
    ax.set_title("SQP trajectory refinement")
    ax.grid(alpha=0.2)
    # Keep the legend below the plotting area so it cannot obscure a pose.
    fig.legend(*ax.get_legend_handles_labels(), loc="lower center",
               bbox_to_anchor=(0.5, 0.015), ncol=3, fontsize=9)

    if cost_ax is not None:
        cost_line, = cost_ax.plot([], [], "o-", color="C1", markersize=3)
        cost_dot, = cost_ax.plot([], [], "o", color="C3", markersize=7)
        cost_ax.set_xlim(-0.5, max(K - 1, 1) + 0.5)
        finite_costs = objective[np.isfinite(objective)]
        if finite_costs.size:
            lo, hi = finite_costs.min(), finite_costs.max()
            padding = max(0.05 * (hi - lo), 0.01 * abs(hi), 1e-6)
            cost_ax.set_ylim(lo - padding, hi + padding)
        cost_ax.set_xlabel("SQP iteration (0 = initial guess)")
        cost_ax.set_ylabel("QP objective (including penalties)")
        cost_ax.set_title("Recorded subproblem objective")
        cost_ax.grid(alpha=0.2)
    fig.tight_layout(rect=(0, 0.10, 1, 1))

    frames = np.unique(np.r_[np.arange(0, K, iteration_stride), K - 1])
    pose_indices = np.unique(np.linspace(0, N - 1, min(num_poses, N), dtype=int))
    pose_artists = []

    def update(frame):
        k = int(frames[frame])
        for artist in pose_artists:
            artist.remove()
        pose_artists.clear()
        current_line.set_data(states[k, :, 0], states[k, :, 1])
        previous = states[max(0, k - 1)]
        previous_line.set_data(previous[:, 0], previous[:, 1])
        previous_line.set_visible(k > 0)
        endpoint.set_data([states[k, -1, 0]], [states[k, -1, 1]])
        for i in pose_indices:
            u = controls[k, i] if controls is not None and i < controls.shape[1] else None
            before = set(ax.get_children())
            plot_rocket(ax, states[k, i], control=u, rocket_length=rocket_length,
                        thrust_limit=thrust_limit, transparency=rocket_alpha)
            for artist in set(ax.get_children()) - before:
                if artist in ax.collections and artist.get_zorder() < 3:
                    artist.remove()  # Ground is already drawn once.
                else:
                    pose_artists.append(artist)
        position_error = np.linalg.norm(states[k, -1, :2] - target[:2])
        speed_error = np.linalg.norm(states[k, -1, 3:5] - target[3:5])
        angle_error = abs(np.arctan2(np.sin(states[k, -1, 2] - target[2]),
                                     np.cos(states[k, -1, 2] - target[2])))
        status.set_text(
            f"SQP iteration {k} / {K - 1}" + (" (initial guess)" if k == 0 else "")
            + f"\nLanding position error: {position_error:.3g} m"
            + f"\nLanding velocity error: {speed_error:.3g} m/s"
            + f"\nLanding angle error: {np.degrees(angle_error):.2f}°"
        )
        artists = [current_line, previous_line, endpoint, status, *pose_artists]
        if cost_ax is not None:
            cost_line.set_data(np.arange(k + 1), objective[:k + 1])
            cost_dot.set_data([k], [objective[k]])
            artists.extend((cost_line, cost_dot))
        return artists

    anim = animation.FuncAnimation(
        fig, update, frames=len(frames), init_func=lambda: update(0),
        interval=interval, blit=False, repeat=repeat,
    )
    if show:
        try:
            return HTML(anim.to_jshtml())
        finally:
            plt.close(fig)
    plt.close(fig)
    return anim


Example of using the plotting and animation functions

In [ ]:
# example of plotting rocket given state and controls
fig, ax = plt.subplots(figsize=(10, 10))
state = jnp.array([1.0, 3.0, 0.1, 0.0, 0.0, 0.0])
control = jnp.array([-50., 100., 50.0])
plot_rocket(
    ax, state, control, transparency=0.2
)
plt.grid(alpha=0.3)

In [ ]:
# example of plotting rocket trajectory given state and control sequence

N = 100
t = np.linspace(0.0, 1.0, N)

states_demo = np.tile(np.array(state), (N, 1))
states_demo[:, 0] = np.linspace(state[0], 0.0, N)   # p_x
states_demo[:, 1] = np.linspace(state[1], 0.2, N)   # p_z
states_demo[:, 2] = np.linspace(state[2], 0.0, N)   # theta

controls_demo = np.zeros((N, 3))
controls_demo[:, 0] = np.linspace(control[0], 0.0, N)   # T_base_x
controls_demo[:, 1] = np.linspace(control[1], 80.0, N)  # T_base_z
controls_demo[:, 2] = np.linspace(control[2], 0.0, N)   # T_nose_x

fig, ax = plt.subplots(figsize=(7, 7))
plot_rocket_trajectory(
    states_demo,
    None,
    ax=ax,
    stride=15,
    alpha=0.1,
    color="C0",
)
plt.show()

In [ ]:
# example of how to animate rocket trajectory (takes a few seconds to run)
animate_rocket_trajectory(
    states_demo, controls_demo,
    dt=0.1,
    rocket_length=2,
)

Helper functions to simulate system given initial state and control sequence. Run the cells as is.

In [ ]:
# run cell as is.
# helper functions for simulation
@eqx.filter_jit
def closed_loop_sim(
    dynamics_discrete: dynamaxsys.Dynamics,
    initial_state: jnp.ndarray,
    controller: Callable[[jnp.ndarray], jnp.ndarray],
    sim_steps: int,
):
    """Simulate the closed-loop system given a closed-loop controller using jax.lax.scan"""

    def step_fn(state, _):
        control = controller(state)
        next_state = dynamics_discrete(state, control)
        return next_state, (next_state, control)

    # Dummy scan inputs as we need sim_steps steps
    dummy_inputs = jnp.arange(sim_steps)
    final_state, (all_states, all_controls) = jax.lax.scan(
        step_fn, initial_state, dummy_inputs
    )

    # Include the initial state
    states = jnp.concatenate([initial_state[None], all_states], axis=0)
    return states, all_controls


@eqx.filter_jit
def open_loop_sim(
    dynamics_discrete: dynamaxsys.Dynamics,
    initial_state: jnp.ndarray,
    controls: jnp.ndarray,
):
    """Simulate the open-loop system given a dynamics model using jax.lax.scan"""

    def step_fn(state, control):
        next_state = dynamics_discrete(state, control)
        return next_state, next_state

    # Run scan
    final_state, all_next_states = jax.lax.scan(step_fn, initial_state, controls)

    # Concatenate initial state at the front
    states = jnp.concatenate([initial_state[None], all_next_states], axis=0)
    return states


### First thing first, we need to define our system dynamics.
Here, we provide you with the nonlinear continuous time dynamics of the rocket. In practice, you, or someone else in your team/company will need to produce the dynamics of the system somehow. Either deriving via first principles, or learning it from data.

In this notebook, we use a custom dynamics package using JAX called `dynamaxsys`, which provides the needed functionality of converting continuous time dynamics into discrete-time dynamics using integration.

#### A little bit more information about the chosen dynamics.
The rocket is modeled similarly to an inverted cart-pole system, but with modifications that allow forces to act in both the horizontal and vertical directions, as well as additional thrusters that generate moments for stabilization.

The full derivation of the rocket dynamics—based on the Euler-Lagrange formalism—can be found [here](https://uw-ctrl.github.io/lmc-book/homework/rocketderivation.html).
If you are not yet familiar with Euler-Lagrange equations, don't worry! You may come across it more if you are interested in designing robots and need to model their dynamics.



In [ ]:
# run cell as is.
class RocketDynamics(dynamaxsys.Dynamics):
    """Rocket dynamics class.
    This class describes the continuous-time dynamics of a rocket with thrust vectoring.
    See the Rocket dynamics derivation for the equations of motion, and how the state and controls are defined.
    """

    def __init__(self, params: Dict[str, float]):
        """
        Initialize the rocket dynamics class.
        Args:
            params: Dictionary containing the rocket parameters.
        """
        state_dim = 6
        control_dim = 3

        def rocket_ode(
            states: jnp.ndarray,
            controls: jnp.ndarray,
            disturbance: jnp.ndarray = None,  # need to provide disturbance for the dynamics class, but not used in this notebook
            time: float = 0.0,  # need to provide time for the dynamics class, but not used in this notebook
        ):
            mass_thruster = params["mass_thruster"]
            mass_body = params["mass_body"]
            length = params["length"]
            gravity = params["gravity"]
            _, _, theta, v_x, v_z, omega = states

            mass = mass_thruster + mass_body

            # First derivatives (kinematic equations)
            velocities = jnp.array([v_x, v_z, omega])

            # Mass matrix M(q)
            M = jnp.array(
                [
                    [mass, 0.0, 0.5 * mass_body * length * jnp.cos(theta)],
                    [0.0, mass, -0.5 * mass_body * length * jnp.sin(theta)],
                    [
                        0.5 * mass_body * length * jnp.cos(theta),
                        -0.5 * mass_body * length * jnp.sin(theta),
                        (1.0 / 3.0) * mass_body * length**2,
                    ],
                ]
            )
            # Coriolis matrix C(q, \dot{q})
            C = jnp.array(
                [
                    [0.0, 0.0, -0.5 * mass_body * length * omega * jnp.sin(theta)],
                    [0.0, 0.0, -0.5 * mass_body * length * omega * jnp.cos(theta)],
                    [0.0, 0.0, 0.0],
                ]
            )

            # gravity vector g(q)
            g_vec = jnp.array(
                [
                    0.0,
                    mass * gravity,
                    -0.5 * mass_body * gravity * length * jnp.sin(theta),
                ]
            )

            # control input Bu with thrust vectoring
            B = jnp.array(
                [
                    [1.0, 0.0, 1.0],
                    [0.0, 1.0, 0.0],
                    [0.0, 0.0, length * jnp.cos(theta)],
                ]
            )
            Bu = B @ controls

            return jnp.concatenate(
                [velocities, jnp.linalg.solve(M, Bu - C @ velocities - g_vec)]
            )

        super().__init__(rocket_ode, state_dim, control_dim)


### Apply a PD Controller for Rocket Landing

In this section, we will use a Proportional-Derivative (PD) controller to stabilize and land the rocket on a specified landing pad. The rocket's state vector is:
$$
x =
\begin{bmatrix}
p_x & p_z & \theta & \dot{p}_x & \dot{p}_z & \dot{\theta}
\end{bmatrix}^T
$$
We independently control the following:
- Horizontal position ($p_x$)
- Vertical position/altitude ($p_z$)
- Orientation ($\theta$)

These are mapped to the rocket's three available thrust inputs:
$$
u =
\begin{bmatrix}
T_{\text{base}, x} \\
T_{\text{base}, z} \\
T_{\text{nose}, x}
\end{bmatrix}
$$
The target state corresponds to the rocket being at the landing pad at the origin, upright ($\theta = 0$), and at rest:
$$
x_\text{target} =
\begin{bmatrix}
0 \\
0 \\
0 \\
0 \\
0 \\
0
\end{bmatrix}
=
\begin{bmatrix}
p_{\text{target}, x} \\
p_{\text{target}, z} \\
\theta_{\text{target}} \\
\dot{p}_{\text{target}, x} \\
\dot{p}_{\text{target}, z} \\
\dot{\theta}_{\text{target}}
\end{bmatrix}
$$
The PD control laws for each input are:
- $T_{\text{base}, x} = -K_{p,x} (p_x - p_{\text{target},x}) - K_{d,x} (\dot{p}_x - \dot{p}_{\text{target},x})$
- $T_{\text{base}, z} = -K_{p,z} (p_z - p_{\text{target},z}) - K_{d,z} (\dot{p}_z - \dot{p}_{\text{target},z})$
- $T_{\text{nose}, x} = -K_{p,\theta} (\theta - \theta_{\text{target}}) - K_{d,\theta} (\dot{\theta} - \dot{\theta}_{\text{target}})$


Pay attention to the thrust limits for each actuator:
- $|T_{\text{base}, x}| \leq T_{\text{base}, x, \max}$: Horizontal thruster can operate bidirectionally.
- $0 \leq T_{\text{base}, z} \leq T_{\text{base}, z, \max}$: Vertical thruster only produces upward thrust (against gravity).
- $|T_{\text{nose}, x}| \leq T_{\text{nose}, \max}$: Nose thruster can operate bidirectionally.


Implement the PD controller as described above and run the following cells to simulate and visualize the system's response.


In [ ]:
def PD_controller(
    state: jnp.ndarray, target: jnp.ndarray, gains: jnp.ndarray, limits: jnp.ndarray
):
    """Compute bounded PD thrust commands for the rocket.

    Args:
        state: Current state [p_x, p_z, theta, v_x, v_z, omega], shape (6,).
        target: Desired state in the same order, shape (6,).
        gains: [Kp_x, Kd_x, Kp_z, Kd_z, Kp_theta, Kd_theta], shape (6,).
        limits: Pair (lower, upper), each with shape (3,), in thrust order.

    Returns:
        JAX array [T_x, T_z, T_theta] of shape (3,), containing the
        horizontal base, vertical base, and horizontal nose thrusts.
    """
    p_x, p_z, theta, v_x, v_z, omega = state
    p_x_target, p_z_target, theta_target, v_x_target, v_z_target, omega_target = target
    Kp_x, Kd_x, Kp_z, Kd_z, Kp_theta, Kd_theta = gains
    lower, upper = limits

    # TODO: Complete the controller using the PD laws in the preceding cell.
    # 1. Compute T_x from the horizontal position and velocity errors,
    #    using Kp_x and Kd_x.
    T_x = -Kp_x * (p_x - p_x_target) - Kd_x * (v_x - v_x_target)

    # 2. Compute T_z from the vertical position and velocity errors,
    #    using Kp_z and Kd_z.
    T_z = -Kp_z * (p_z - p_z_target) - Kd_z * (v_z - v_z_target)
    
    # 3. Compute T_theta (the nose thrust) from the orientation and angular
    #    velocity errors, using Kp_theta and Kd_theta.
    #    Use current minus target for each error and negative feedback.
    T_theta = -Kp_theta * (theta - theta_target) - Kd_theta * (omega - omega_target)
    
    # 4. Use jnp.clip to bound each command by its corresponding entries
    #    in lower and upper: index 0 for T_x, 1 for T_z, and 2 for T_theta.
    #    In particular, the vertical thruster cannot produce downward thrust.
    # Replace the exception below with your implementation. Use JAX operations
    # so this function remains compatible with the simulation helpers.
    # abs(T_x) <= upper[0]
    # 0 <= T_z <= upper[1]
    # T_theta <= upper[2]
    
    jnp.clip(T_x, lower[0], upper[0])
    jnp.clip(T_z, lower[1], upper[1])
    jnp.clip(T_theta, lower[2], upper[2])

    return jnp.array([T_x, T_z, T_theta])


Now, set up the dynamics and other parameters. We will be using the helper function `closed_loop_sim` to simulate the dynamics given a closed-loop controller.

In [ ]:
# feel free to adjust the parameters to see how they affect the rocket's behavior
rocket_length = 1.0  # length of the rocket
params = {
    "mass_thruster": 10.0,  # mass of the thruster
    "mass_body": 15,  # mass of the rocket body
    "length": rocket_length,  # length of the rocket
    "gravity": 9.81,  # gravity constant
}

# set up the continuous-time dynamics and convert to discrete-time dynamics
rocket_ct = RocketDynamics(params)
dt = 0.01
rocket_dt = dynamaxsys.get_discrete_time_dynamics(
    rocket_ct, dt=dt
)  # discrete time dynamics

# set up the control limits
thrust_to_weight_ratio = 1.3
rocket_weight = (params["mass_thruster"] + params["mass_body"]) * params["gravity"]
T_base_z_max = rocket_weight * thrust_to_weight_ratio  # vertical thrust limit
T_base_x_max = T_base_z_max * 07.5  # horizontal thrust limit; feel free to adjust this
T_nose_max = T_base_x_max * 07.5  # nose thrust limit; feel free to adjust this

# set up the lower and upper limits for the control inputs
lower = np.array(
    [-T_base_x_max, 0.0, -T_nose_max]
)  # lower limits for the control inputs
upper = np.array(
    [T_base_x_max, T_base_z_max, T_nose_max]
)  # upper limits for the control inputs
limits = [lower, upper]


Now, select some gains for your controller, simulate, and plot the results!

NOTE: Try some gains, but the point is that it is not easy to find gains that perfectly land the rocket. So don't spend all night tuning the gains. Just try for 5 minutes.

In [ ]:
# Run this cell after completing PD_controller. No code needs to be filled in here.
# Experiment by changing gains, initial_state, and n_steps below,
# then rerun the whole cell.

target_state_param = jnp.array([0.0, 0.0, 0.0, 0.0, 0.0, 0.0])  # target state
# ADJUST: PD gains, ordered as
# [Kp_x, Kd_x, Kp_z, Kd_z, Kp_theta, Kd_theta].
# Each pair controls horizontal position, altitude, or orientation, respectively.
gains = jnp.array(
    [50.0, 30.0, 50.0, 150.0, 90.0, 100.0]
)

# Run as provided: bind the current gains, target, and actuator limits.
# Rerun this setup whenever you change gains.
controller = functools.partial(
    PD_controller, target=target_state_param, gains=gains, limits=limits
)

# ADJUST: starting state, ordered as [p_x, p_z, theta, v_x, v_z, omega].
# Positions are in meters, theta in radians, linear velocities in m/s,
# and angular velocity in rad/s. Start with p_z > 0 (above the ground).
rocket_initial_state = jnp.array(
    [0.2, 10.0, 0.1, 0.2, -0.1, -0.05]
)
# ADJUST: n_steps is the number of simulation steps.
# Increase it to simulate for longer or decrease it for a shorter run.
# The total simulation duration is n_steps * dt seconds.
n_steps = 300

# Run the simulation and plotting code below as provided.
states, controls = closed_loop_sim(
    rocket_dt, rocket_initial_state, controller, n_steps
)  # simulate the dynamics given a closed-loop controller

# plot the trajectory
fig, ax = plt.subplots(figsize=(10,10))
plot_rocket_trajectory(
    states,
    None,
    ax=ax,
    stride=25,
    alpha=0.1,
    color="C0",
)



In [ ]:
# animate the trajectory. This takes a few seconds to run.
animate_rocket_trajectory(
    states, controls,
    dt=0.1,
    rocket_length=rocket_length,
    thrust_limit=upper
)

### Land the rocket with SQP!

You may have noticed that tuning a good set of PD gains can be difficult, and even an effective gain set may only succeed for some initial conditions. The challenge of reliably landing the rocket motivates a more systematic approach.
Trajectory optimization allows us to compute the most efficient sequence of controls and states, while explicitly enforcing constraints to prevent the rocket from crashing and to satisfy other requirements.

Specifically, we aim to find a sequence of states $\{x_0, x_1, \ldots, x_N\}$ and controls $\{u_0, u_1, \ldots, u_{N-1}\}$ that minimizes the total weighted state and control effort, subject to the rocket ending at the origin (both in position, velocity, and angle), obeying the system dynamics, and respecting input limits:

$$
\begin{align*}
\text{minimize}_{x_{0:N},\, u_{0:N-1}} \quad & \sum_{k=0}^{N-1} x_k^\top Q x_k + u_k^\top R u_k \\
\text{subject to} \quad
     & x_{k+1} = f(x_k, u_k), \quad\;\; k = 0, \ldots, N-1 \\
     & x_0 = x_\text{init} \\
     & x_N = x_\text{target} \\
     & p_{z,k} \geq 0, \qquad k = 0, \ldots, N\\
     & u_\text{min} \leq u_k \leq u_\text{max}, \qquad k = 0, \ldots, N-1 \\
\end{align*}
$$

Here, $x_k$ is the full rocket state at step $k$, and $u_k$ is the control input at step $k$.
However, because the dynamics $f(x, u)$ are highly nonlinear, directly solving this optimization is extremely challenging.
In the next steps, you will implement the essential components for solving this problem via Sequential Quadratic Programming (SQP).


#### Linearize dynamics

Notice in the trajectory optimization problem above that the objective is quadratic, and all constraints except for the dynamics are linear. This means the problem is almost a quadratic program (QP)!

To fully convert it into a QP, we need to linearize the dynamics constraint.

Implement the following function, which outputs $A$, $B$, and $C$ satisfying $\dot{x} \approx Ax + Bu + C$ at a chosen operating point.

*Hint: We are using JAX, so and differentiation can be performed by calling a function. You don't need to do any differentiation by hand!*

In [ ]:
@eqx.filter_jit
def linearize_dynamics(
    dynamics: Callable[[jnp.ndarray, jnp.ndarray], jnp.ndarray],
    state: jnp.ndarray,
    control: jnp.ndarray,
    eval_time: float = 0.0,
):
    """Construct an affine approximation of the supplied dynamics.

    Args:
        dynamics: Callable accepting state, control, and the keyword time.
            Its output has shape (n,).
        state: Operating-point state, shape (n,).
        control: Operating-point control, shape (m,).
        time: Evaluation time; hold fixed when differentiating.

    Returns:
        A: Jacobian with respect to state, shape (n, n).
        B: Jacobian with respect to control, shape (n, m).
        C: Affine offset, shape (n,).
    """
    # 1. Compute A and B by differentiating dynamics with respect to its state
    #    and control arguments, then evaluate at the supplied operating point.
    #    Hint: jax.jacobian supports differentiation with respect to multiple
    #    arguments using argnums. Keep time fixed and pass it as time=time.
    jacobian = jax.jacobian(dynamics, argnums=(0, 1))
    A, B = jacobian(state, control, time=eval_time)
    
    # 2. Evaluate dynamics at the same state, control, and time. Determine C
    #    so the affine approximation A @ x + B @ u + C exactly matches that
    #    output at the operating point (x, u) = (state, control).
    dynamics_at_time = dynamics(state, control, time=eval_time)
    C = dynamics_at_time - (A @ state + B @ control)

    # 3. Return A, B, C with the shapes listed above. Use JAX operations and
    #    the supplied arguments; do not rely on a global timestep variable.
    return A, B, C


In [ ]:
state = jnp.array([0.2, 10.0, 0.1, 0.2, -0.1, -0.05])
control = jnp.array([100.0, 200.0, -50.0])
A, B, C = linearize_dynamics(rocket_dt, state, control)
A, B, C


Next, let's linearize the dynamics along the state and control trajectory produced by your PD controller.
This process will yield linear, time-varying dynamics of the form:
$$
x_{k+1} = A_k x_k + B_k u_k + C_k
$$
where $A_k$, $B_k$, and $C_k$ are computed at each timestep $k$ along the trajectory.

These matrices capture the local, linear approximation of the true nonlinear dynamics at each point in time along the trajectory.
Thus, the system is described by *time-varying* linear dynamics.

To efficiently compute $A_k$, $B_k$, and $C_k$ for all timesteps, apply `jax.vmap` to `linearize_dynamics`.

In [ ]:
# Run this simulation as provided to obtain the PD controller trajectory.
states, controls = closed_loop_sim(rocket_dt, rocket_initial_state, controller, n_steps)

# Compute As, Bs, Cs along this trajectory using one line of JAX code.
# Apply jax.vmap to your linearize_dynamics function:
# - Keep the dynamics callable rocket_dt fixed across all timesteps.
# - Map over matching state and control pairs (x_k, u_k), k = 0, ..., N-1.
# - states contains N+1 entries, whereas controls contains N entries. Exclude
#   the terminal state when selecting the states at which to linearize.
# Hint: use in_axes to specify which arguments are fixed and which are batched.
# Assign the three returned arrays to As, Bs, Cs. Their expected shapes are
# (N, 6, 6), (N, 6, 3), and (N, 6), respectively, where N = n_steps.
# As, Bs, Cs = ...
# Replace the exception below with your single-line implementation.
As, Bs, Cs = jax.vmap(linearize_dynamics, in_axes=(None, 0, 0))(rocket_dt, jnp.array(states[:-1]), jnp.array(controls))

plot_rocket_trajectory(states, controls)
# plot_linearization(As, Bs, Cs, states, controls)
# plt.show()
# [print("A: ", As[i], ", B: ", Bs[i], ", C: ", Cs[i]) for i in range(len(As))]


#### Set up and solve the CVXPY problem
Having linearized the dynamics around a given trajectory, we can now formulate and solve a trajectory optimization problem using this linearized system.

The locally linearized, discrete-time dynamics are given by:

$$
\begin{align*}
x_{k+1} &= A_k x_k + B_k u_k + C_k, \qquad k = 0, ..., N-1 \\
\end{align*}
$$

where $A_k$, $B_k$, $C_k$ are the time-varying linearization matrices about $(\tilde{x}_k, \tilde{u}_k)$.

For the linearized system, we formulate the following quadratic program (QP):

$$
\begin{align*}
\underset{x_{0:N},\ u_{0:N-1}}{\text{minimize}} \quad
    & \sum_{k=0}^{N-1} \left(
        x_k^\top Q x_k
        + u_k^\top R u_k
        + (x_k - \tilde{x}_k)^\top Q_\mathrm{trp} (x_k - \tilde{x}_k)
        + (u_k - \tilde{u}_k)^\top R_\mathrm{trp} (u_k - \tilde{u}_k)
      \right) \\
    &\quad + (x_N - x_\mathrm{target})^\top Q_N (x_N - x_\mathrm{target}) \\
\text{subject to} \qquad
    & x_{k+1} = A_k x_k + B_k u_k + C_k, \quad   k = 0, ..., N-1 \\
    & x_0 = x_\mathrm{init} \\
    & p_{z,k} \geq 0, \qquad k = 0, \ldots, N\\
    & u_{\min} \leq u_k \leq u_{\max}, \qquad k = 0, ..., N-1 \\
\end{align*}
$$

Here, the matrices $Q_\mathrm{trp}$ and $R_\mathrm{trp}$ impose a *trust region penalty*, encouraging the optimized trajectory $(x_k, u_k)$ to stay close to the trajectory $(\tilde{x}_k, \tilde{u}_k)$ about which the dynamics were linearized. This is important because the quality of the linear approximation degrades further from the linearization points.

Additionally, rather than enforcing a hard terminal state constraint (which might be infeasible for a given horizon or control limits), we use a large terminal cost via $Q_N$ to softly encourage the final state to reach the desired target.
With these design choices, the problem becomes a convex quadratic program (QP) which we can efficiently solve using `cvxpy`.

In the following cells, we set up the corresponding `cvxpy` problem as described above. Fill any any code as directed.

In [ ]:
# Run this cell as provided first; no code needs to be filled in.
# Then experiment with n_steps and the cost weights below.
# Keep state_dim and control_dim unchanged: these are determined by the model.
state_dim = rocket_ct.state_dim  # 6
control_dim = rocket_ct.control_dim  # 3

# ADJUST: prediction horizon in steps (duration = n_steps * dt seconds).
# A longer horizon gives more time to land but increases the QP size.
# After changing n_steps, rerun the following setup cells to rebuild the QP
# and regenerate trajectories with matching lengths.
n_steps = 300

# ADJUST: diagonal entries of Q weight the running state cost.
# State order: [p_x, p_z, theta, v_x, v_z, omega].
# Larger entries penalize the corresponding state more strongly at each step.
# A zero entry means no direct running penalty on that component.
Q = cp.Constant(
    np.diag([1.0, 0, 20.0, 0.5, 0.5, 10.0])
)

# ADJUST: diagonal entries of R penalize control effort at each step.
# Control order: [horizontal base thrust, vertical base thrust, nose thrust].
# Larger entries discourage use of the corresponding thruster.
R = cp.Constant(np.diag([0.0001, 0.0001, 0.00]))

# ADJUST: QN weights the terminal error relative to the target state.
# It uses the same state order as Q. You may change the individual diagonal
# entries or the overall multiplier (100) to emphasize the final landing state.
QN = (
    cp.Constant(np.diag([100.0, 500.0, 1000.0, 1000.0, 800.0, 1000.0])) * 100
)

# ADJUST: the scalar multiplier (10.0) in QTRP penalizes deviations from
# the reference state trajectory used for linearization.
# Larger values discourage large changes in states between SQP iterations.
QTRP = cp.Constant(
    np.eye(state_dim) * 10.0
)

# ADJUST: the scalar multiplier (10.0) in RTRP penalizes deviations from
# the reference control trajectory used for linearization.
# Larger values discourage large changes in controls between SQP iterations.
RTRP = cp.Constant(
    np.eye(control_dim) * 10.0
)

# Keep all diagonal weights and multipliers nonnegative so the QP stays convex.
# To compare cost weights, keep n_steps fixed and change one choice at a time.
# After changing any weights, rerun this cell and the following QP setup/solve
# cells so the new constants are included in the optimization problem.


In [ ]:
# set up cvxpy optimization variables
states_cvx = cp.Variable((n_steps + 1, state_dim),name="states")  # state trajectory [n_steps+1, state_dim]
controls_cvx = cp.Variable((n_steps, control_dim),name="controls")  # control trajectory [n_steps, control_dim]

# set up the A, B, C matrices as parameters since they will be updated at each SQP iteration
As = cp.Parameter((n_steps, state_dim, state_dim), name="As")
Bs = cp.Parameter((n_steps, state_dim, control_dim), name="Bs")
Cs = cp.Parameter((n_steps, state_dim), name="Cs")

# set up the previous state and control parameters since they will be updated at each SQP iteration
previous_states = cp.Parameter((n_steps + 1, state_dim), name="previous_states")
previous_controls = cp.Parameter((n_steps, control_dim), name="previous_controls")

# set up the goal state as a parameter in case you want to change the goal state later on
target_state_param = cp.Parameter((state_dim), name="target_state")

# set up the initial state parameter in case you want to change the initial state later on
initial_state_param = cp.Parameter(state_dim, name="initial_state")

# set up control constraints
u_min = cp.Constant(lower, name="u_min")
u_max = cp.Constant(upper, name="u_max")

# TODO: Build the objective and constraints using the definitions above.
# Implement the QP described in the preceding Markdown cell.
#
# 1. Initialize a scalar objective and a list of constraints. Enforce that
#    the first state equals the initial_state parameter.
objective = 0
constraints = []
constraints.append(initial_state_param == states_cvx[0])


constraints.append(controls_cvx[:,0] >= u_min[0])
constraints.append(controls_cvx[:,0] <= u_max[0])
constraints.append(controls_cvx[:,1] >= u_min[1])
constraints.append(controls_cvx[:,1] <= u_max[1])
constraints.append(controls_cvx[:,2] >= u_min[2])
constraints.append(controls_cvx[:,2] <= u_max[2])
constraints.append(states_cvx[:,1] >= 0.0)

for i in range(n_steps - 1):
    # 2. For each control timestep k = 0, ..., n_steps-1, add these costs:
    #    - Running state cost, weighted by Q.
    #    - Control effort, weighted by R.
    #    - State deviation from previous_states[k], weighted by QTRP.
    #    - Control deviation from previous_controls[k], weighted by RTRP.
    #    Hint: cp.quad_form(vector, weight_matrix) expresses a quadratic cost.
    objective += cp.quad_form(states_cvx[i], Q)
    objective += cp.quad_form(controls_cvx[i], R)
    objective += cp.quad_form((states_cvx[i] - previous_states[i]), QTRP)
    objective += cp.quad_form((controls_cvx[i] - previous_controls[i]), RTRP)
    
    # 3. At each of these timesteps, add constraints for:
    #    - The affine dynamics, using As[k], Bs[k], and Cs[k] to relate states[k],
    #      controls[k], and states[k+1]. Include the affine offset Cs[k].
    #    - Both lower and upper input bounds, using u_min and u_max.
    #    - Nonnegative altitude (state component 1).
    slack = 0.0065
    constraints.append(cp.abs(((As[i] @ states_cvx[i]) + (Bs[i] @ controls_cvx[i]) + Cs[i]) - states_cvx[i + 1]) <= slack)

# 4. After the loop, add the terminal tracking cost relative to target_state,
#    weighted by QN, and enforce nonnegative altitude at the terminal state.
#    Use a terminal cost rather than a hard terminal equality constraint.
#
objective += cp.quad_form((states_cvx[i] - target_state_param), QN)
objective += cp.maximum(-states_cvx[i][1], 0.0) * 10000000

# Use CVXPY expressions directly; parameter values are assigned in later cells.
# Do not solve the problem here. The line below constructs it once you have
# defined objective and constraints.
objective.name = "objective"
problem = cp.Problem(cp.Minimize(objective), constraints)




Now, compute and set the values for the Parameters and solve!

NOTE: `cvxpy` doesn't like `jax.numpy` arrays. So we need to convert any arrays into `np.array` for `cvxpy`.

In [ ]:
# Run this cell after completing the linearization and QP setup cells.
# No code needs to be filled in here. Adjust target_state_value and
# initial_state_value to experiment, then rerun this cell and the following
# parameter-assignment, solve, and visualization cells.
# Both arrays use [p_x, p_z, theta, v_x, v_z, omega]: positions in meters,
# theta in radians, linear velocities in m/s, and angular velocity in rad/s.

# ADJUST: the desired terminal state for the optimization.
# The default represents landing at the origin, upright and at rest.
target_state_value = np.array(
    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
)

# ADJUST: the rocket's starting position, orientation, and velocities.
# Start above the ground (p_z > 0) and explore different initial conditions.
initial_state_value = np.array(
    [1.0, 10.0, 0.1, -0.5, -0.1, -0.05]
)

# Run as provided: use the PD controller to generate the initial guess.
# This uses the gains and PD target previously bound to controller. Changing
# target_state_value above changes the QP target, not the PD controller target.
# To change the PD gains or target, edit and rerun the earlier controller cell.
previous_states_values, previous_controls_values = closed_loop_sim(
    rocket_dt, initial_state_value, controller, n_steps
)

# Run as provided: linearize along the initial guess with your implementation.
# These arrays supply the dynamics parameter values in the following cell.
As_values, Bs_values, Cs_values = jax.vmap(linearize_dynamics, in_axes=(None, 0, 0))(
    rocket_dt, previous_states_values[:-1], previous_controls_values
)


In [ ]:
# set the values for the parameters. Convert to numpy array to avoid issues with cvxpy.
As.value = np.array(As_values)
Bs.value = np.array(Bs_values)
Cs.value = np.array(Cs_values)

previous_states.value = np.array(previous_states_values)
previous_controls.value = np.array(previous_controls_values)

target_state_param.value = target_state_value
initial_state_param.value = initial_state_value


# solve the problem
problem.solve(solver=cp.CLARABEL)

# Debug draw
# fig_test, ax_text = plt.subplots(figsize=(7, 7))
# temp = [(state_cvx[0], state_cvx[1], state_cvx[2], state_cvx[3], state_cvx[4], state_cvx[5]) for state_cvx in states_cvx]
# plot_rocket_trajectory(
#     temp,
#     controls_cvx.value,
#     ax=ax,
#     stride=15,
#     alpha=0.1,
#     color="C0",
# )
# plt.show()

if problem.status not in (cp.OPTIMAL, cp.OPTIMAL_INACCURATE):
    report = cp_debug.DebugReport(problem)
    cp_debug.debug(problem)
    cp_debug.debug_infeasibility(problem, report)
    cp_debug.debug_numerical_issues(problem, report)
    cp_debug.debug_unboundedness(problem, report)
# else:
#     report = cp_debug.DebugReport(problem)
#     cp_debug.debug(problem)


In [ ]:
# compute states with true nonlinear dynamics. Recall that states.value is the optimized trajectory based on linearized dynamics.
# What we want is the optimized controls applied to the true nonlinear dynamics.
# We can use the open_loop_sim function to compute the states with true nonlinear dynamics.
open_loop_states = open_loop_sim(rocket_dt, initial_state_value, controls_cvx.value)

In [ ]:
# plot the trajectory
fig, ax = plt.subplots(figsize=(10,10))
plot_rocket_trajectory(
    open_loop_states,
    controls_cvx.value,
    rocket_length=rocket_length,
    thrust_limit=upper,
    ax=ax,
    stride=25,
    alpha=0.1,
    color="C0",
)


In [ ]:
# animate the trajectory. This takes a few seconds to run.
animate_rocket_trajectory(
    open_loop_states, controls_cvx.value,
    dt=0.1,
    rocket_length=rocket_length,
    thrust_limit=upper
)

#### Perform Sequential Quadratic Programming (SQP) Iterations
So far, you have completed a single SQP iteration. The power of SQP comes from repeating this process:
linearizing around the current trajectory, solving the optimization, and then updating the trajectory based on the new solution.

Implement a loop to carry out multiple SQP iterations. After each iteration, use the latest solution (states/controls) as the new nominal trajectory about which to linearize for the next step. Continue this process to progressively improve the trajectory.

In [ ]:
# setting up the problem again

# target state value
target_state_value = np.array(
    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
)  # feel free to adjust target state

# initial state value
initial_state_value = np.array(
    [0.1, 3.0, 0.1, -0.1, -0.1, -0.05]
)  # feel free to adjust initial state

# previous states and controls values using the outputs of the PD controller
previous_states_values, previous_controls_values = closed_loop_sim(
    rocket_dt, initial_state_value, controller, n_steps
)

# set the values for the parameters.
target_state_param.value = target_state_value
initial_state_param.value = initial_state_value

In [ ]:
# ADJUST: number of SQP iterations; start with the default and experiment.
n_SQP_iterations = 10
controls_list = [previous_controls_values]  # list of controls to collect for plotting
states_list = [previous_states_values]  # list of states to collect for plotting
costs = []  # list of costs to collect for plotting

# loop through SQP iterations
for i in range(n_SQP_iterations):
    print(f"SQP Iteration {i + 1}/{n_SQP_iterations}")

    # 1. Assign previous_states_values and previous_controls_values to the
    #    corresponding CVXPY reference parameters. Convert to np.array when
    #    assigning parameter values; CVXPY expects NumPy arrays, not JAX arrays.
    previous_states.value = np.array(previous_states_values)
    previous_controls.value = np.array(previous_controls_values)

    # 2. Use your vectorized linearization from the earlier exercise to compute
    #    As_values, Bs_values, Cs_values along the current reference trajectory.
    #    Pair each control with its state and exclude the terminal state.
    As_values, Bs_values, Cs_values = jax.vmap(linearize_dynamics, in_axes=(None, 0, 0))(
        rocket_dt,
        jnp.array(previous_states_values[:-1]),
        jnp.array(previous_controls_values)
    )

    # 3. Assign these arrays to the .value attributes of As, Bs, and Cs,
    #    converting to NumPy arrays as needed.
    As.value = np.array(As_values)
    Bs.value = np.array(Bs_values)
    Cs.value = np.array(Cs_values)

    # 4. Solve the existing problem with the CLARABEL solver. Do not rebuild
    #    the CVXPY problem inside the loop; only its parameter values change.
    #    Check problem.status before using the solution. If no solution was
    #    returned, raise an informative error rather than using None values.
    problem.solve(solver=cp.CLARABEL)
    if problem.status not in (cp.OPTIMAL, cp.OPTIMAL_INACCURATE):
        report = cp_debug.DebugReport(problem)
        cp_debug.debug(problem)
        cp_debug.debug_infeasibility(problem, report)
        cp_debug.debug_numerical_issues(problem, report)
        cp_debug.debug_unboundedness(problem, report)
        raise RuntimeError("Could not find a solution at iteration ", i)

    # 5. Record the QP objective value in costs. Use the optimized state and
    #    control trajectories as previous_states_values and
    #    previous_controls_values for the next iteration.
    costs.append(problem.value)
    # previous_states_values, previous_controls_values = closed_loop_sim(rocket_dt, initial_state_value, controller, n_steps)
    # open_loop_states = open_loop_sim(rocket_dt, initial_state_value, controls_cvx.value)
    previous_states_values = states_cvx.value
    previous_controls_values = controls_cvx.value

    # 6. Append the optimized state and control sequence to controls_list and
    #    states_list so the provided code below can simulate and visualize
    #    progress across SQP iterations.
    states_list.append(states_cvx.value)
    controls_list.append(controls_cvx.value)

controls_list = np.array(controls_list)

# with the collected controls, compute the states with true nonlinear dynamics.
# Recall that states.value is the optimized trajectory based on linearized dynamics, so they don't match the true nonlinear dynamics.
states_list = jax.vmap(open_loop_sim, in_axes=(None, None, 0))(
    rocket_dt, initial_state_value, controls_list
)

In [ ]:
# animate the last trajectory from the SQP iterations
animate_rocket_trajectory(states_list[-1], controls_list[-1], thrust_limit=upper)

In [ ]:
# Watch the complete trajectory change across SQP iterations.
animate_sqp_iterations(
    states_list, None, costs=costs,
    rocket_length=rocket_length, thrust_limit=upper,
    target_state=target_state_value,
    interval=400,
)


#### Add obstacle constraints

Suppose that there is a circular obstacle located at $(x_0, z_0)$ with radius $r$ that the rocket needs to avoid. The constraint is
$$
g(x= [p_x, p_z, \theta, \dot{p}_x, \dot{p}_x, \dot{\theta}]^T) = r^2 - (p_x - x_0)^2 - (p_z - z_0)^2  \leq 0
$$
We require this constraint to be satisfied at every time step along the trajectory.
This is a nonlinear, non-convex constraint. Like what we did with the dynamics, we can also linearize the obstacle constraint about a state $\tilde{x}_k$

$$
g(x_k) \approx g(\tilde{x}_k) + \nabla g(\tilde{x}_k)^T(x_k - \tilde{x}_k) = G_kx_k + h_k
$$

Thus the QP problem we wish to solve at each SQP iteration becomes

$$
\begin{align*}
\text{minimize}_{x_{0:N}, u_{0:N-1}} \ & \left[\sum_{k=0}^{N-1} x_k^TQx_k + u_k^TRu_k + (x_k - \tilde{x}_k)^TQ_\text{tpr}(x_k - \tilde{x}_k) + (u_k - \tilde{u}_k)^TR_\text{tpr}(u_k - \tilde{u}_k) \right]  + (x_N - x_{\text{target}})^T Q_N (x_N - x_{\text{target}}) \\
\text{subject to} \quad
& x_{k+1} = A_k x_k + B_k u_k + C_k, \qquad k = 0, ..., N-1 \\
& x_0 = x_{\text{init}} \\
& p_{z,k} \geq 0, \qquad k = 0, \ldots, N\\
& G_kx_k + h_k \leq 0 \\
& u_{\text{min}} \leq u_k \leq u_{\text{max}}, \qquad k = 0, ..., N-1 \\
\end{align*}
$$


Copy your `cvxpy` problem developed the the previous part and modify it here to include the linearized obstacle constraint. Then run multiple SQP iterations.


In [ ]:
# obstacle constraint function
def circular_obstacle_constraint(
    state: jnp.ndarray, obs_center: jnp.ndarray, obs_radius: float
) -> jnp.ndarray:
    """Compute obstacle constraint value"""
    p_x, p_z = state[0], state[1]
    obs_x, obs_z = obs_center
    dist_sq = (p_x - obs_x) ** 2 + (p_z - obs_z) ** 2
    return obs_radius**2 - dist_sq


In [ ]:
state_dim = rocket_ct.state_dim  # 6, state dimension
control_dim = rocket_ct.control_dim  # 3, control dimension
n_steps = 300  # feel free to adjust number of steps.

# set up the cost matrices. Feel free to adjust these to see how they affect the optimization.
Q = cp.Constant(
    np.diag([1.0, 0, 20.0, 0.5, 0.5, 10.0]), name="state_cost"
)  # encourage the rocket to stay upright and slow
R = cp.Constant(
    np.diag([0.005, 0.001, 0.00]), name="control_cost"
)  # control cost matrix.
QN = cp.Constant(
    np.diag([100.0, 500.0, 1000.0, 1000.0, 800.0, 1000.0]) * 100, name="terminal_cost"
)  # terminal cost matrix.
QTRP = cp.Constant(
    np.eye(state_dim) * 5.0, name="state_trust_region_penalty"
)  # trust region penalty for state deviations.
RTRP = cp.Constant(
    np.eye(control_dim) * 5.0, name="control_trust_region_penalty"
)  # trust region penalty for control deviations.


In [ ]:
# TODO: Create a new CVXPY problem that includes the circular obstacle.
# 1. Copy your completed QP setup from the earlier exercise: optimization
#    variables, dynamics/reference parameters, initial/target state parameters,
#    input bounds, objective, and constraints. Recreate these objects here
#    using the same variable names so the following cells can use them.
#    Keep the original running, trust-region, and terminal costs, along with
#    the initial condition, affine dynamics, input bounds, and altitude limits.
#
states_cvx = cp.Variable((n_steps + 1, state_dim),name="states")  # state trajectory [n_steps+1, state_dim]
controls_cvx = cp.Variable((n_steps, control_dim),name="controls")  # control trajectory [n_steps, control_dim]
As = cp.Parameter((n_steps, state_dim, state_dim), name="As")
Bs = cp.Parameter((n_steps, state_dim, control_dim), name="Bs")
Cs = cp.Parameter((n_steps, state_dim), name="Cs")
previous_states = cp.Parameter((n_steps + 1, state_dim), name="previous_states")
previous_controls = cp.Parameter((n_steps, control_dim), name="previous_controls")
target_state_param = cp.Parameter((state_dim), name="target_state")
initial_state_param = cp.Parameter(state_dim, name="initial_state")
u_min = cp.Constant(lower, name="u_min")
u_max = cp.Constant(upper, name="u_max")

objective = 0
constraints = []
constraints.append(initial_state_param == states_cvx[0])
constraints.append(controls_cvx[:,0] >= u_min[0])
constraints.append(controls_cvx[:,0] <= u_max[0])
constraints.append(controls_cvx[:,1] >= u_min[1])
constraints.append(controls_cvx[:,1] <= u_max[1])
constraints.append(controls_cvx[:,2] >= u_min[2])
constraints.append(controls_cvx[:,2] <= u_max[2])
constraints.append(states_cvx[:,1] >= 0.0)

for i in range(n_steps - 1):
    objective += cp.quad_form(states_cvx[i], Q)
    objective += cp.quad_form(controls_cvx[i], R)
    objective += cp.quad_form((states_cvx[i] - previous_states[i]), QTRP)
    objective += cp.quad_form((controls_cvx[i] - previous_controls[i]), RTRP)
    
    slack = 0.0065
    constraints.append(cp.abs(((As[i] @ states_cvx[i]) + (Bs[i] @ controls_cvx[i]) + Cs[i]) - states_cvx[i + 1]) <= slack)

objective += cp.quad_form((states_cvx[i] - target_state_param), QN)
objective += cp.maximum(-states_cvx[i][1], 0.0) * 10000000
objective.name = "objective"
problem = cp.Problem(cp.Minimize(objective), constraints)

# 2. Define the additional CVXPY parameters for the obstacle linearization:
#    - Gs: shape (n_steps + 1, state_dim), the gradient at each reference state.
#    - hs: shape (n_steps + 1,), the affine offset at each reference state.
#    Use cp.Parameter: their numerical values change at each SQP iteration.
#    Also define obstacle_center (shape (2,)) and obstacle_radius (scalar,
#    shape ()) as cp.Parameter objects for the setup cell that follows.
#    The nonlinear obstacle function will be evaluated and differentiated
#    with JAX outside the CVXPY problem, using the chosen center and radius.
#
Gs = cp.Parameter((n_steps + 1, state_dim))
hs = cp.Parameter(n_steps + 1)
obstacle_center = cp.Parameter(2)
obstacle_radius = cp.Parameter()

# 3. Use Gs and hs to add the affine obstacle inequality from the preceding
#    Markdown cell at EVERY state timestep, including the terminal state.
#    Each row of Gs multiplies the corresponding full state vector; hs provides
#    its scalar offset. Check the inequality direction against the definition
#    of circular_obstacle_constraint (nonpositive means outside the circle).
#    Do not put the nonlinear circular constraint directly into CVXPY.
#
for i in range(n_steps):
    # circle_penetration = circular_obstacle_constraint(states_cvx[i], obstacle_center, obstacle_radius)
    slack = 0.0
    constraints.append(Gs[i] @ states_cvx[i] + hs[i] <= slack)

# 4. Define objective and constraints for this new problem. Parameter values
#    and their SQP updates are handled in later cells; do not solve here.
#    Keep the problem-construction line below and avoid copying it twice.
problem = cp.Problem(cp.Minimize(objective), constraints)


In [ ]:
# setting up the problem again

# target state value
target_state_value = np.array(
    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
)  # feel free to adjust target state

# choose obstacle center and radius
obstacle_center_value = np.array([-0.01, 4.0])  # feel free to adjust obstacle center
obstacle_radius_value = np.array(0.5)  # feel free to adjust obstacle radius

# obstacle constraint function. Use functools.partial to fix the obstacle center and radius.
obstacle_constraint = functools.partial(
    circular_obstacle_constraint,
    obs_center=obstacle_center_value,
    obs_radius=obstacle_radius_value,
)

# initial state value
initial_state_value = np.array([0.1, 8.0, 0.1, -0.1, -0.1, -0.05])

# previous states and controls values using the outputs of the PD controller
previous_states_values, previous_controls_values = closed_loop_sim(
    rocket_dt, initial_state_value, controller, n_steps
)


target_state_param.value = target_state_value
initial_state_param.value = initial_state_value

# update the obstacle center and radius
obstacle_center.value = obstacle_center_value
obstacle_radius.value = obstacle_radius_value


In [ ]:
# TODO: Adapt your completed SQP loop to the problem with an obstacle.
# Copy your earlier loop, including its iteration-count setting, initialization
# of controls_list and costs, and conversion of controls_list to a NumPy array
# after the loop. Keep the nonlinear rollout code below as provided.

def collision_constraints(state, state_guess, center, radius):
    away = state_guess[:2] - center
    n_hat = away / np.maximum(np.linalg.norm(away, axis=1, keepdims=True), 1e-9)
    return cp.sum(cp.multiply(n_hat, state[:2] - center), axis=1) >= d

n_SQP_iterations = 10
controls_list = [previous_controls_values]  # list of controls to collect for plotting
states_list = [previous_states_values]  # list of states to collect for plotting
costs = []  # list of costs to collect for plotting

for i in range(n_SQP_iterations):
    print(f"SQP Iteration {i + 1}/{n_SQP_iterations}")
    # 1. Update the reference state/control parameters and linearized dynamics
    #    parameters just as before, using the previous optimized trajectory.
    previous_states.value = np.array(previous_states_values)
    previous_controls.value = np.array(previous_controls_values)

    # 2. Linearize obstacle_constraint about every reference state, INCLUDING the
    #    terminal state. Use JAX automatic differentiation to obtain the gradient
    #    with respect to state, and jax.vmap to evaluate it along the trajectory.
    #    Store the gradients in Gs_values, shape (n_steps + 1, state_dim).
    As_values, Bs_values, Cs_values = jax.vmap(linearize_dynamics, in_axes=(None, 0, 0))(
        rocket_dt,
        jnp.array(previous_states_values[:-1]),
        jnp.array(previous_controls_values)
    )
    As.value = np.array(As_values)
    Bs.value = np.array(Bs_values)
    Cs.value = np.array(Cs_values)

    grad_func = jax.grad(obstacle_constraint)
    Gs_values = jax.vmap(grad_func, in_axes=(0))(jnp.array(previous_states_values))

    # 3. Determine hs_values so that each affine approximation matches the original
    #    obstacle function at its reference state. Evaluate the obstacle function
    #    there and subtract the gradient's dot product with that same state.
    #    Use batched JAX operations; hs_values must have shape (n_steps + 1,).
    #    Unlike the dynamics linearization, do not omit the terminal state here.
    def calculate_h(reference_state_k, Gs_k):
        return obstacle_constraint(reference_state_k) - Gs_k @ reference_state_k

    hs_values = jax.vmap(calculate_h, in_axes=(0, 0))(
        jnp.array(previous_states_values),
        Gs_values
    )

    # 4. Assign Gs_values and hs_values to the obstacle parameters Gs.value and
    #    hs.value, converting to NumPy arrays. Update all parameters before solving.
    Gs.value = np.array(Gs_values)
    hs.value = np.array(hs_values)

    # 5. Solve the existing CVXPY problem and check its status before using the
    #    solution. If no solution is returned, raise an informative error.
    report = cp_debug.DebugReport(problem)
    problem.solve(solver=cp.CLARABEL)
    if problem.status not in (cp.OPTIMAL, cp.OPTIMAL_INACCURATE):
        report = cp_debug.DebugReport(problem)
        cp_debug.debug(problem)
        cp_debug.debug_infeasibility(problem, report)
        cp_debug.debug_numerical_issues(problem, report)
        cp_debug.debug_unboundedness(problem, report)
        raise RuntimeError("Could not find a solution at iteration ", i)

    # 6. Record the objective in costs, use the optimized states and controls as
    #    the next reference trajectory, and append the controls to controls_list.
    costs.append(problem.value)
    previous_states_values = states_cvx.value
    previous_controls_values = controls_cvx.value

    states_list.append(states_cvx.value)
    controls_list.append(controls_cvx.value)

controls_list = np.array(controls_list)

#
# Do not rebuild the CVXPY problem inside the loop. Use obstacle_constraint
# from the preceding setup cell; its center and radius are already fixed.

# compute states with true nonlinear dynamics. Recall that states.value is the optimized trajectory based on linearized dynamics.
states_list = jax.vmap(open_loop_sim, in_axes=(None, None, 0))(
    rocket_dt, initial_state_value, controls_list
)

In [ ]:
animate_rocket_trajectory(
    states_list[-1],
    controls_list[-1],
    obstacle_center=obstacle_center_value,
    obstacle_radius=obstacle_radius_value,
    thrust_limit=upper,
)

In [ ]:
# Watch the complete trajectory change across SQP iterations.
animate_sqp_iterations(
    states_list, controls_list, costs=costs,
    rocket_length=rocket_length, thrust_limit=upper,
    target_state=target_state_value,
    obstacle_center=obstacle_center_value,
    obstacle_radius=obstacle_radius_value,
    interval=400,
)
